# Financial Wavelet Analysis

This notebook is a **source-faithful reconstruction of the Python code printed in Appendix A of the final thesis** *Análise de séries temporais multivariadas via Wavelet*.

The final application analyzes IBOVESPA, Dow Jones Industrial Average, S&P 500 and Bitcoin. Typographic quote characters introduced by PDF/LaTeX rendering were normalized to valid Python syntax, but the original analytical choices were preserved.

### Important source notes

The final thesis contains a few internal inconsistencies that are intentionally not silently corrected here:

- the text describes closing prices, while the appendix code concatenates `bitcoin["Open"]`;
- the methodology text describes linear interpolation, while the appendix code uses `interpolate(method="backfill")`;
- the CWT subsection is labeled "IBOVESPA" in the appendix, while the code assigns `df["DJIA"]` to the analyzed series.


## 1. Imports and market data


In [ ]:
# Original appendix setup
# !pip install yfinance --upgrade --no-cache-dir

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader.data as web
import datetime
import missingno as msno
import pywt
import yfinance as yf

yf.pdr_override()

ibov = web.get_data_yahoo("^BVSP")
dji = web.get_data_yahoo("^DJI")
SeP500 = web.get_data_yahoo("^GSPC")
bitcoin = web.get_data_yahoo("BTC-USD")

df_ = pd.concat(
    [ibov["Close"], dji["Close"], SeP500["Close"], bitcoin["Open"]],
    axis=1,
    join="outer",
)
df_.columns = ["IBOV", "DJIA", "SeP500", "bitcoin"]
df_o = df_["2012-01-01":"2023-08-04"]
df_o.tail()


## 2. Missing-data inspection and treatment


In [ ]:
msno.matrix(df_o[["IBOV", "DJIA", "bitcoin", "SeP500"]])
msno.bar(df_o[["IBOV", "DJIA", "bitcoin", "SeP500"]])

ax = df_o.IBOV["2022-01-01":"2023-01-01"].plot(figsize=(18, 10))
ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento (em reais)", fontsize=20)
plt.title("Preço de fechamento do BOVESPA de 2022 até 2023", fontsize=30)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()

df_o.IBOV = df_o.IBOV.interpolate(method="backfill")
df_o.DJIA = df_o.DJIA.interpolate(method="backfill")
df_o.SeP500 = df_o.SeP500.interpolate(method="backfill")
df_o.bitcoin = df_o.bitcoin.interpolate(method="backfill")

df = df_o.copy()


## 3. Stationarity diagnostics


In [ ]:
from statsmodels.tsa.stattools import adfuller

def test_stationarity(timeseries, NOME):
    rolmean = timeseries.rolling(window=30).mean()
    rolstd = timeseries.rolling(window=30).std()

    plt.plot(timeseries, label=NOME)
    plt.plot(rolmean, label="Média móvel")
    plt.plot(rolstd, label="Var móvel")
    plt.legend(loc="best")
    plt.title("Média e variância móveis")
    plt.show(block=False)

    print("Resultado do teste de Dickey-Fuller:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput[f"Critical Value ({key})"] = value
    print(dfoutput)


## 4. Financial-series visualization


In [ ]:
ax = df[["IBOV"]].plot(figsize=(18, 10))
ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento (em reais)", fontsize=20)
plt.title("Preço de fechamento do Índice IBOVESPA", fontsize=30)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()
test_stationarity(df["IBOV"], "IBOVESPA")

ax = df[["DJIA"]].plot(figsize=(18, 10))
ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento (em dólares)", fontsize=20)
plt.title("Preço de fechamento do Dow Jones", fontsize=30)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()
test_stationarity(df["DJIA"], "DJIA")

ax = df.bitcoin.plot(figsize=(18, 10))
ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento (em dólares)", fontsize=20)
plt.title("Preço de fechamento do Bitcoin", fontsize=30)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()
test_stationarity(df["bitcoin"], "Bitcoin")

ax = df.SeP500.plot(figsize=(18, 10))
ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento (em dólares)", fontsize=20)
plt.title("Preço de fechamento do SP500", fontsize=30)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()
test_stationarity(df["SeP500"], "S&P500")


In [ ]:
ax = df["DJIA"].plot(figsize=(18, 10))
df["IBOV"].plot()
df["bitcoin"].plot()
df["SeP500"].plot()

ax.set_xlabel("Data", fontsize=20)
ax.set_ylabel("Preço de fechamento", fontsize=20)
plt.title(
    "Preços de fechamento do Dow Jones, BOVESPA, S&P500 e Bitcoin",
    fontsize=30,
)
plt.legend(loc="upper left", fontsize=20)
plt.tight_layout()
plt.grid(True)
plt.show()


## 5. Continuous Wavelet Transform and scalogram

The appendix subsection is labeled **IBOVESPA**, but the original code analyzes `df["DJIA"]`. This reconstruction preserves that assignment.


In [ ]:
serie_d = df["DJIA"].copy()

amostras_pad = 200
serie_pad = pywt.pad(serie_d, amostras_pad, "symmetric")

ano_inicial = min(serie_d.index).year
ano_final = max(serie_d.index).year

anos = np.arange(ano_inicial, ano_final + 1, 1)
ia = np.arange(0, len(serie_d), 330)

wav = "gaus1"
s = np.arange(1, 100)

coef, freqs = pywt.cwt(serie_pad, s, wav, method="fft")
coef_DJIA = np.abs(coef) ** 2
coef_DJIA = coef_DJIA[:, amostras_pad:-amostras_pad]

plt.figure(dpi=150, figsize=(10, 4))

plt.subplot(121)
plt.plot(serie_d)
plt.ylabel("Dow Jones Index - Preço de fechamento")
plt.xlabel("Ano")
plt.grid()

plt.subplot(122)
plt.imshow(
    coef_DJIA,
    extent=[0, len(serie_d), s[-1], s[0]],
    cmap="jet",
    aspect="auto",
    interpolation="spline16",
)
plt.xticks(ia, anos, rotation="vertical")
plt.xlabel("Translação ($\\tau$)")
plt.ylabel("Escala ($s$)")
plt.colorbar()
plt.grid(False)
plt.tight_layout()
plt.show()


## 6. 3D visualization of wavelet coefficients


In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

coef_df = pd.DataFrame(coef_DJIA)
df_3d = coef_df.unstack().reset_index()
df_3d.columns = ["X", "Y", "Z"]

df_3d["X"] = pd.Categorical(df_3d["X"])
df_3d["X"] = df_3d["X"].cat.codes

datas = pd.Series(df.index.to_list())
datas_ = []

for j in range(len(datas)):
    for i in range(0, 99):
        datas_.append(datas[j])

df_3d["Date"] = datas_

fig = plt.figure(figsize=(15, 16))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(
    df_3d["Y"],
    df_3d["X"],
    df_3d["Z"],
    cmap="jet",
    linewidth=0.2,
)
ax.set_xlabel("$Escala$", fontsize=15)
ax.set_ylabel("$t$", fontsize=15)
ax.set_zlabel("$Coeficientes TWC$", fontsize=15, rotation=0)
ax.view_init(30, 10)
